In [15]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'C:\Users\AniyaBagheri\GF500_research_perliminary_data_analysis\comparative analysis\Primary_Historic_CSV.csv',
    encoding='latin-1'
)
print(df.columns)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
pivot_data = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for year in years:
        year_data = company_data[company_data['year'] == year]
        if not year_data.empty:
            row[f'region_{year}'] = year_data['region'].iloc[0]
            row[f'sector_{year}'] = year_data['sector '].iloc[0]
            row[f're100_{year}'] = year_data['re100'].iloc[0]
            row[f'sbti_{year}'] = year_data['sbti'].iloc[0]
            row[f'cn_{year}'] = year_data['cn'].iloc[0]
            row[f'nz_{year}'] = year_data['nz'].iloc[0]
            row[f'cc_{year}'] = year_data['cc'].iloc[0]
    
    pivot_data.append(row)

matrix = pd.DataFrame(pivot_data)
matrix.to_csv('company_matrix.csv', index=False)

print(unique_companies)



Index(['year', 'company ', 'region', 'sector ', 're100', 'sbti', 'cn', 'nz',
       'cc'],
      dtype='object')
['3m' 'abb' 'abbott laboratories' 'abbvie' 'accenture' 'achmea' 'acs'
 'aegon' 'aeon' 'agricultural bank of china' 'aia group' 'airbus' 'aisin'
 'albertsons' 'alfresa holdings' 'alibaba group holding'
 'alimentation couche-tard' 'allianz' 'allstate' 'alphabet'
 'aluminum corp. of china' 'amazon' 'amer international group'
 'américa móvil' 'american express' 'american international group'
 'amerisourcebergen' 'amgen' 'anglo american' 'anheuser-busch inbev'
 'anhui conch group' 'ansteel group' 'anthem' 'apple' 'arcelormittal'
 'archer daniels midland' 'arrow electronics' 'assicurazioni generali'
 'astrazeneca' 'at&t' 'aviation industry corp. of china' 'aviva' 'axa'
 'bae systems' 'banco bilbao vizcaya argentaria' 'banco bradesco'
 'banco do brasil' 'banco santander' 'bank of america' 'bank of china'
 'bank of communications' 'bank of montreal' 'bank of nova scotia'
 'barclays'

In [16]:
print(unique_companies)
print(len(unique_companies))


['3m' 'abb' 'abbott laboratories' 'abbvie' 'accenture' 'achmea' 'acs'
 'aegon' 'aeon' 'agricultural bank of china' 'aia group' 'airbus' 'aisin'
 'albertsons' 'alfresa holdings' 'alibaba group holding'
 'alimentation couche-tard' 'allianz' 'allstate' 'alphabet'
 'aluminum corp. of china' 'amazon' 'amer international group'
 'américa móvil' 'american express' 'american international group'
 'amerisourcebergen' 'amgen' 'anglo american' 'anheuser-busch inbev'
 'anhui conch group' 'ansteel group' 'anthem' 'apple' 'arcelormittal'
 'archer daniels midland' 'arrow electronics' 'assicurazioni generali'
 'astrazeneca' 'at&t' 'aviation industry corp. of china' 'aviva' 'axa'
 'bae systems' 'banco bilbao vizcaya argentaria' 'banco bradesco'
 'banco do brasil' 'banco santander' 'bank of america' 'bank of china'
 'bank of communications' 'bank of montreal' 'bank of nova scotia'
 'barclays' 'basf' 'bayer' 'beijing automotive group'
 'beijing jianlong heavy industry group' 'berkshire hathaway' 'best bu

In [17]:
print(company_data)

      year                  company          region  \
2489  2025  GuideWell Mutual Holding  North America   

                              sector   re100  sbti  cn  nz  cc  \
2489  Health care and pharmaceuticals    NaN   NaN NaN NaN NaN   

            company_normalized         company_canonical  
2489  guidewell mutual holding  guidewell mutual holding  


In [18]:
import pandas as pd
from difflib import get_close_matches

df = pd.read_csv(
    r'Primary_Historic_CSV.csv',
    encoding='latin-1'
)

df['company_normalized'] = df['company '].str.strip().str.lower()

unique_companies = df['company_normalized'].unique()
name_mapping = {}

for company in unique_companies:
    matches = get_close_matches(company, unique_companies, n=5, cutoff=0.85)
    if len(matches) > 1:
        canonical = min(matches, key=len)
        for match in matches:
            name_mapping[match] = canonical
    else:
        name_mapping[company] = company

df['company_canonical'] = df['company_normalized'].map(name_mapping)

years = sorted(df['year'].unique())
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']
results = []

for company in df['company_canonical'].unique():
    company_data = df[df['company_canonical'] == company]
    row = {'company': company}
    
    for target in targets:
        transition = []
        for year in years:
            year_data = company_data[company_data['year'] == year]
            if year_data.empty:
                transition.append('-')
            else:
                val = year_data[target].iloc[0]
                if pd.isna(val):
                    transition.append('0')
                elif val == 1:
                    transition.append('1')
                elif val == -1:
                    transition.append('-1')
                else:
                    transition.append('0')
        row[target] = ''.join(transition)
    
    results.append(row)

output = pd.DataFrame(results)
output.to_csv('company_transitions_map_cc_fixed.csv', index=False)



# SBTI to net zero

In [12]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# Total lost SBTi
lost_sbti = 0
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            lost_sbti += 1
            break

print(f"Lost SBTi: {lost_sbti}")

# Lost SBTi and gained NZ or CN
lost_sbti_gained = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    nz = row['nz'].replace('-', '')
    cn = row['cn'].replace('-', '')
    
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            if '1' in nz[i+1:] or '1' in cn[i+1:]:
                lost_sbti_gained.append({
                    'company': row['company'],
                    'sbti': row['sbti'],
                    'nz': row['nz'],
                    'cn': row['cn']
                })
            break

result = pd.DataFrame(lost_sbti_gained)
result.to_csv('sbti_lost_then_gained.csv', index=False)
print(f"Lost SBTi and gained NZ/CN: {len(result)}")

print(lost_sbti)

Lost SBTi: 15
Lost SBTi and gained NZ/CN: 10
15


# carbon neutral to net zero

In [8]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

lost_cn_gained_nz = []
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            if '1' in nz[i+1:]:
                lost_cn_gained_nz.append({'company': row['company'], 'cn': row['cn'], 'nz': row['nz']})
            break

result = pd.DataFrame(lost_cn_gained_nz)
result.to_csv('lost_cn_gained_nz.csv', index=False)
print(result)

lost_cn = 0
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            break

print(f"Lost CN: {lost_cn}")

                             company     cn     nz
0                                abb  11-00  00-10
1                               aeon  10000  01110
2              alibaba group holding  01110  00001
3           alimentation couche-tard  00010  00001
4                            allianz  11000  11110
..                               ...    ...    ...
159  contemporary amperex technology  --010  --001
160                  lufthansa group  --110  --001
161                    tongwei group  --110  --001
162      luxshare precision industry  --110  --001
163                 societe generale  ---10  ---01

[164 rows x 3 columns]
Lost CN: 199


In [ ]:
# Track CN 2021 -> NZ 2025
cn_to_nz = matrix[
    (matrix['cn_2021'] == 1) & 
    (matrix['nz_2025'] == 1)
]['company'].tolist()

# Track SBTi changes every two years
sbti_transitions = {}
for i in range(len(years) - 1):
    year1, year2 = years[i], years[i+1]
    if year2 - year1 <= 2:
        gained = matrix[
            (matrix[f'sbti_{year1}'] != 1) & 
            (matrix[f'sbti_{year2}'] == 1)
        ]['company'].tolist()
        lost = matrix[
            (matrix[f'sbti_{year1}'] == 1) & 
            (matrix[f'sbti_{year2}'] != 1)
        ]['company'].tolist()
        sbti_transitions[f'{year1}_to_{year2}'] = {'gained': gained, 'lost': lost}

print(f"Companies with CN in 2021 and NZ in 2025: {len(cn_to_nz)}")
print(f"\nSBTi transitions: {sbti_transitions}")

matrix.to_csv('company_matrix.csv', index=False)

that brings the 

In [3]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

print(df.iloc[489:639])
for i in range(5):
    nz = (df['nz'].str[i] == '1').sum()
    cn = (df['cn'].str[i] == '1').sum()
    print(f"{2021+i}: NZ={nz}, CN={cn}")

                           company  re100   sbti     cn     nz     cc
489              sinochem holdings  -000-  -000-  -000-  -000-  -000-
490            mercedes-benz group  -0000  -0011  -1110  -0001  -1010
491                elevance health  -1111  -0001  -0000  -0110  -0101
492                 meta platforms  -1110  -0011  -0000  -1110  -0011
493  life insurance corp. of india  -0000  -0000  -0000  -0000  -0000
..                             ...    ...    ...    ...    ...    ...
632   international airlines group  ----0  ----0  ----0  ----0  ----1
633   pnc financial services group  ----0  ----0  ----0  ----0  ----1
634      perusahaan listrik negara  ----0  ----0  ----0  ----0  ----1
635              st. james's place  ----0  ----0  ----0  ----0  ----1
636       guidewell mutual holding  ----0  ----0  ----0  ----0  ----0

[148 rows x 6 columns]
2021: NZ=120, CN=146
2022: NZ=182, CN=157
2023: NZ=197, CN=109
2024: NZ=221, CN=97
2025: NZ=89, CN=52


## dealing with normalization and false exits/entries:





#### Category 1: Company Rebrands (7 cases)

1. Royal Dutch Shell → Shell

Old name: royal dutch shell (2019-2021)
New name: shell (2022-2025)
Change year: 2021
Impact on data: Shell data starts in 2022 with NZ commitment

2. Facebook → Meta Platforms

Old name: facebook (2019-2021)
New name: meta platforms (2022-2025)
Change year: October 2021
Impact on data: Meta shows sustainability commitments starting 2022

3. Daimler → Mercedes-Benz Group

Old name: daimler (2019-2021)
New name: mercedes-benz group (2022-2025)
Change year: 2022
Impact on data: Mercedes-Benz shows comprehensive commitments from 2023+

4. GlaxoSmithKline → GSK

Old name: glaxosmithkline (2019-2021)
New name: gsk (2022-2025)
Change year: July 2022
Impact on data: GSK continues strong commitments from earlier period

5. Raytheon Technologies → RTX

Old name: raytheon technologies (2019-2022)
New name: rtx (2023-2025)
Change year: July 2023
Impact on data: RTX shows NZ commitment starting 2024

6. ViacomCBS → Paramount Global

Old name: viacomcbs (2019-2021)
New name: paramount global (2022-2025)
Change year: February 2022
Impact on data: Both show no commitments (all zeros/dashes)

7. Anthem → Elevance Health

Old name: anthem (2019-2021)
New name: elevance health (2022-2025)
Change year: June 2022
Impact on data: Elevance shows multiple commitments starting 2022


Category 2: Holding Company Structures (4 cases)
8. Panasonic / Panasonic Holdings

Pattern: Original "Panasonic" (2019-2021) → "Panasonic Holdings" (2022-2025)
Reason: Holding company restructuring in 2022
Impact: Panasonic Holdings shows continued commitments

9. POSCO / POSCO Holdings

Pattern: "POSCO" (2019-2022) → "POSCO Holdings" (2023-2025)
Reason: Holding company restructuring
Impact: Some data overlap in transition years

10. SK / SK Group

Pattern: Both names appear across different years
Reason: Holding company structure
Impact: Need to merge these records

11. Sinochem / Sinochem Holdings

Pattern: "Sinochem" (2019-2021) → "Sinochem Holdings" (2022-2024)
Reason: Holding company restructuring
Impact: All zeros across both entries


Category 3: Accent Variations (4 cases)
12. Nestlé / Nestle

Pattern: "Nestlé" (2019-2023) and "Nestle" (2024-2025)
Reason: Database encoding differences
Impact: SAME COMPANY - data should be continuous

13. América Móvil / America Movil

Pattern: Both spellings used
Reason: Accent handling in data entry
Impact: SAME COMPANY - merge needed

14. Raízen / Raizen

Pattern: Both spellings used
Reason: Accent handling
Impact: SAME COMPANY - no commitments in either

15. Société Générale / Societe Generale

Pattern: Both spellings used
Reason: Accent handling
Impact: SAME COMPANY - some CN commitments


Category 4: Abbreviations (2 cases)
16. Nippon Telegraph and Telephone / NTT

Full name: nippon telegraph and telephone (2019-2023)
Abbreviation: ntt (2025)
Impact: SAME COMPANY - continuous operations

17. International Business Machines / IBM

Full name: international business machines (2019-2022)
Abbreviation: ibm (2023-2025)
Impact: SAME COMPANY - IBM shows NZ commitments from 2024


Category 5: Name Simplifications (2 cases)
18. Deutsche Post DHL Group / DHL Group

Old name: deutsche post dhl group (2019-2022)
New name: dhl group (2023-2025)
Impact: Name simplification, continuous operations

19. PKN Orlen Group / Orlen

Old name: pkn orlen group (2019-2022)
New name: orlen (2023-2025)
Impact: Name simplification


Category 6: Mergers & Acquisitions (1 case)
20. Synnex → TD Synnex

Old name: synnex (2019-2021)
New name: td synnex (2022-2025)
Merger: Tech Data Corporation merged with Synnex in 2021
Impact: TD Synnex shows commitments starting 2023


Category 7: Company Splits (1 case)
21. General Electric Split

Original: general electric (2019-2023)
Split into:

general electric (ge aerospace) (2024-2025)
ge vernova (2025)


Split year: 2024
Impact: GE split into three companies (Healthcare spun off as GE HealthCare in 2023, then GE split into Aerospace and Vernova in 2024)


Category 8: Other Variations (6 cases)
22. Mitsubishi / Mitsubishi Corp

Pattern: Both names used
Impact: Likely different entities (Mitsubishi Corporation vs Mitsubishi Group)
Note: May actually be different companies

23. Nippon Steel / Nippon Steel Corporation

Pattern: "Nippon Steel Corporation" (2019-2023), "Nippon Steel" (2025)
Impact: SAME COMPANY - name variation

24. Bunge / Bunge Global

Pattern: "Bunge" (2019-2023), "Bunge Global" (2025)
Impact: Name change to Bunge Global in 2024

25. China COSCO Shipping / COSCO Shipping

Pattern: "China COSCO Shipping" (2019-2021), "COSCO Shipping" (2022-2025)
Impact: Name simplification

26. Olam International / Olam Group

Pattern: "Olam International" (2019-2021), "Olam Group" (2022-2025)
Impact: Restructuring in 2022

27. AmerisourceBergen / Cencora

Old name: amerisourcebergen (2019-2022)
New name: cencora (2023-2025)
Change year: August 2023
Impact: Company rebrand


#### name differences 

High Priority (MUST MERGE)
These are definitely the same company and should be treated as single entities:

Royal Dutch Shell / Shell
Facebook / Meta Platforms
GlaxoSmithKline / GSK
Anthem / Elevance Health
Raytheon Technologies / RTX
Daimler / Mercedes-Benz Group
AmerisourceBergen / Cencora
Nestlé / Nestle (accent variation)
América Móvil / America Movil (accent variation)
Société Générale / Societe Generale (accent variation)
Raízen / Raizen (accent variation)
Nippon Telegraph and Telephone / NTT
International Business Machines / IBM
Deutsche Post DHL Group / DHL Group
PKN Orlen Group / Orlen
Synnex / TD Synnex
Bunge / Bunge Global
China COSCO Shipping / COSCO Shipping
Olam International / Olam Group
Nippon Steel / Nippon Steel Corporation

#### holding company decisions 
Medium Priority (HOLDING COMPANY DECISIONS)
These require a decision on whether to treat as single entity or separate:

Panasonic / Panasonic Holdings
POSCO / POSCO Holdings
SK / SK Group
Sinochem / Sinochem Holdings

Recommendation: Treat as single company for sustainability analysis, as holding companies typically consolidate environmental commitments.

#### company split 
Special Case (COMPANY SPLIT)

General Electric → GE Aerospace + GE Vernova

Recommendation:

Keep pre-2024 data under "General Electric"
Track 2024+ separately for split entities
Note that sustainability commitments may have transferred to new entities

Uncertain (NEED VERIFICATION)

Mitsubishi / Mitsubishi Corp

Recommendation: Verify if these are truly different entities (Mitsubishi Corporation vs Mitsubishi Group companies)


( yes truly diff after verification)

In [15]:
import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# Define rebrands: (old_name, new_name, final_name)
rebrands = [
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
]

# Print comparison
for old, new, final in rebrands:
    old_row = df[df['company'].str.lower() == old].iloc[0]
    new_row = df[df['company'].str.lower() == new].iloc[0]
    
    print(f"\n{old.upper()} → {new.upper()} = {final.upper()}")
    print(f"       RE100  SBTi   CN     NZ     CC")
    print(f"Old:   {old_row['re100']}  {old_row['sbti']}  {old_row['cn']}  {old_row['nz']}  {old_row['cc']}")
    print(f"New:   {new_row['re100']}  {new_row['sbti']}  {new_row['cn']}  {new_row['nz']}  {new_row['cc']}")

# Merge function
def merge_strings(s1, s2):
    return ''.join(c1 if c1 != '-' else c2 for c1, c2 in zip(s1 + '-'*len(s2), s2 + '-'*len(s1)))[:max(len(s1), len(s2))]




ROYAL DUTCH SHELL → SHELL = SHELL
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  0----  1----  0----
New:   -0000  -0000  -0000  -1110  -1011

FACEBOOK → META PLATFORMS = META PLATFORMS
       RE100  SBTi   CN     NZ     CC
Old:   1----  0----  0----  1----  0----
New:   -1110  -0011  -0000  -1110  -0011

DAIMLER → MERCEDES-BENZ GROUP = MERCEDES-BENZ GROUP
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  1----  0----  0----
New:   -0000  -0011  -1110  -0001  -1010

GLAXOSMITHKLINE → GSK = GSK
       RE100  SBTi   CN     NZ     CC
Old:   1----  1----  0----  1----  0----
New:   -0111  -1111  -0000  -1110  -0011

RAYTHEON TECHNOLOGIES → RTX = RTX
       RE100  SBTi   CN     NZ     CC
Old:   00---  00---  00---  00---  00---
New:   --000  --000  --000  --110  --010

VIACOMCBS → PARAMOUNT GLOBAL = PARAMOUNT GLOBAL
       RE100  SBTi   CN     NZ     CC
Old:   0----  0----  0----  0----  0----
New:   -00--  -00--  -00--  -00--  -00--

ANTHEM → ELEVANCE HEALTH = ELEVANC

In [5]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

rebrands = [
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
]

def merge_strings(s1, s2):
    max_len = max(len(s1), len(s2))
    result = []
    for i in range(max_len):
        c1 = s1[i] if i < len(s1) else '-'
        c2 = s2[i] if i < len(s2) else '-'
        result.append(c1 if c1 != '-' else c2)
    return ''.join(result)

merged_rows = []
for old, new, final in rebrands:
    old_row = df[df['company'].str.lower() == old].iloc[0]
    new_row = df[df['company'].str.lower() == new].iloc[0]
    
    merged_rows.append({
        'company': final,
        're100': merge_strings(old_row['re100'], new_row['re100']),
        'sbti': merge_strings(old_row['sbti'], new_row['sbti']),
        'cn': merge_strings(old_row['cn'], new_row['cn']),
        'nz': merge_strings(old_row['nz'], new_row['nz']),
        'cc': merge_strings(old_row['cc'], new_row['cc']),
    })
    
    print(f"{final}: RE100={merged_rows[-1]['re100']} NZ={merged_rows[-1]['nz']}")

old_new_names = [old for old, new, _ in rebrands] + [new for _, new, _ in rebrands]
df_final = df[~df['company'].str.lower().isin([n.lower() for n in old_new_names])]
df_final = pd.concat([df_final, pd.DataFrame(merged_rows)], ignore_index=True)
df_final = df_final.sort_values('company').reset_index(drop=True)

df_final.to_csv('company_data_rebrands_merged.csv', index=False)
print(f"\nOriginal: {len(df)} → Merged: {len(df_final)} (removed {len(df)-len(df_final)} duplicates)")

Shell: RE100=00000 NZ=11111
Meta Platforms: RE100=11110 NZ=11111
Mercedes-Benz Group: RE100=00000 NZ=00000
GSK: RE100=10111 NZ=11111
RTX: RE100=00000 NZ=00110
Paramount Global: RE100=000-- NZ=000--
Elevance Health: RE100=11111 NZ=00111
Cencora: RE100=00000 NZ=00000
TD Synnex: RE100=00000 NZ=01111
Bunge Global: RE100=00000 NZ=00000
COSCO Shipping: RE100=00000 NZ=00111
Olam Group: RE100=00000 NZ=00011

Original: 637 → Merged: 625 (removed 12 duplicates)


In [19]:

import pandas as pd

df = pd.read_csv('company_transitions_map_cc_fixed.csv')

# All merges: (old_name, new_name, final_name)
merges = [
    # Rebrands
    ('royal dutch shell', 'shell', 'Shell'),
    ('facebook', 'meta platforms', 'Meta Platforms'),
    ('daimler', 'mercedes-benz group', 'Mercedes-Benz Group'),
    ('glaxosmithkline', 'gsk', 'GSK'),
    ('raytheon technologies', 'rtx', 'RTX'),
    ('viacomcbs', 'paramount global', 'Paramount Global'),
    ('anthem', 'elevance health', 'Elevance Health'),
    ('amerisourcebergen', 'cencora', 'Cencora'),
    ('synnex', 'td synnex', 'TD Synnex'),
    ('bunge', 'bunge global', 'Bunge Global'),
    ('china cosco shipping', 'cosco shipping', 'COSCO Shipping'),
    ('olam international', 'olam group', 'Olam Group'),
    # Holding companies
    ('panasonic', 'panasonic holdings', 'Panasonic Holdings'),
    ('posco', 'posco holdings', 'POSCO Holdings'),
    ('sk', 'sk group', 'SK Group'),
    ('sinochem', 'sinochem holdings', 'Sinochem Holdings'),
    # Accent variations
    ('nestlé', 'nestle', 'Nestlé'),
    ('américa móvil', 'america movil', 'América Móvil'),
    ('raízen', 'raizen', 'Raízen'),
    ('société générale', 'societe generale', 'Société Générale'),
    # Abbreviations
    ('nippon telegraph and telephone', 'ntt', 'NTT'),
    ('international business machines', 'ibm', 'IBM'),
    # Name simplifications
    ('deutsche post dhl group', 'dhl group', 'DHL Group'),
    ('pkn orlen group', 'orlen', 'Orlen'),
    # Other
    ('nippon steel corporation', 'nippon steel', 'Nippon Steel'),
]

def merge_strings(s1, s2):
    max_len = max(len(s1), len(s2))
    result = []
    for i in range(max_len):
        c1 = s1[i] if i < len(s1) else '-'
        c2 = s2[i] if i < len(s2) else '-'
        result.append(c1 if c1 != '-' else c2)
    return ''.join(result)

merged_rows = []
for old, new, final in merges:
    old_row = df[df['company'].str.lower() == old]
    new_row = df[df['company'].str.lower() == new]
    
    if len(old_row) > 0 and len(new_row) > 0:
        old_row = old_row.iloc[0]
        new_row = new_row.iloc[0]
        
        merged_rows.append({
            'company': final,
            're100': merge_strings(old_row['re100'], new_row['re100']),
            'sbti': merge_strings(old_row['sbti'], new_row['sbti']),
            'cn': merge_strings(old_row['cn'], new_row['cn']),
            'nz': merge_strings(old_row['nz'], new_row['nz']),
            'cc': merge_strings(old_row['cc'], new_row['cc']),
        })

old_new_names = [old for old, new, _ in merges] + [new for _, new, _ in merges]
df_final = df[~df['company'].str.lower().isin([n.lower() for n in old_new_names])]
df_final = pd.concat([df_final, pd.DataFrame(merged_rows)], ignore_index=True)
df_final = df_final.sort_values('company').reset_index(drop=True)

df_final.to_csv('matrix_fixed.csv', index=False)
print(f"{len(df)} → {len(df_final)} ({len(df)-len(df_final)} merged)")
for i in range(5):
    cn = (df_final['cn'].str[i] == '1').sum()
    nz = (df_final['nz'].str[i] == '1').sum()

    print(f"{2021+i}: NZ={nz}, CN={cn}")

637 → 612 (25 merged)
2021: NZ=120, CN=146
2022: NZ=182, CN=157
2023: NZ=197, CN=109
2024: NZ=221, CN=97
2025: NZ=251, CN=89


In [30]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

for idx, row in df.iterrows():
    cn_list = list(row['cn'])
    nz_list = list(row['nz'])
    
    for i in range(len(cn_list)):
        if cn_list[i] == '1' and nz_list[i] == '1':
            cn_list[i] = '0'
    
    df.at[idx, 'cn'] = ''.join(cn_list)

df.to_csv('matrix_fixed.csv', index=False)
print(f"{len(df)} → {len(df_final)} ({len(df)-len(df_final)} merged)")
for i in range(5):
    nz = (df['nz'].str[i] == '1').sum()
    cn = (df['cn'].str[i] == '1').sum()
    print(f"{2021+i}: NZ={nz}, CN={cn}")

612 → 612 (0 merged)
2021: NZ=120, CN=105
2022: NZ=182, CN=103
2023: NZ=197, CN=109
2024: NZ=221, CN=97
2025: NZ=251, CN=89


In [31]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

# Total lost SBTi
lost_sbti = 0
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            lost_sbti += 1
            break

print(f"Lost SBTi: {lost_sbti}")

# Lost SBTi and gained NZ or CN
lost_sbti_gained = []
for _, row in df.iterrows():
    sbti = row['sbti'].replace('-', '')
    nz = row['nz'].replace('-', '')
    cn = row['cn'].replace('-', '')
    
    for i, char in enumerate(sbti):
        if char == '1' and '0' in sbti[i+1:]:
            if '1' in nz[i+1:] or '1' in cn[i+1:]:
                lost_sbti_gained.append({
                    'company': row['company'],
                    'sbti': row['sbti'],
                    'nz': row['nz'],
                    'cn': row['cn']
                })
            break

result = pd.DataFrame(lost_sbti_gained)
result.to_csv('sbti_lost_then_gained.csv', index=False)
print(f"Lost SBTi and gained NZ/CN: {len(result)}")

print(lost_sbti)

rate = len(lost_sbti_gained) / lost_sbti if lost_sbti != 0 else 0
print(f"transition rate: {rate:.4f}")


Lost SBTi: 15
Lost SBTi and gained NZ/CN: 14
15
transition rate: 0.9333


In [32]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

def clean(s):
    return s.replace('-', '')

# Year 1→3
sbti_lost_y13 = 0
sbti_lost_gained_y13 = 0
for _, row in df.iterrows():
    sbti = clean(row['sbti'])[:3]
    nz = clean(row['nz'])[:3]
    cn = clean(row['cn'])[:3]
    if '1' in sbti:
        loss_idx = sbti.index('1')
        if '0' in sbti[loss_idx+1:]:
            sbti_lost_y13 += 1
            if '1' in nz[loss_idx+1:] or '1' in cn[loss_idx+1:]:
                sbti_lost_gained_y13 += 1

# Year 3→5
sbti_lost_y35 = 0
sbti_lost_gained_y35 = 0
for _, row in df.iterrows():
    sbti = clean(row['sbti'])[2:5]
    nz = clean(row['nz'])[2:5]
    cn = clean(row['cn'])[2:5]
    if '1' in sbti:
        loss_idx = sbti.index('1')
        if '0' in sbti[loss_idx+1:]:
            sbti_lost_y35 += 1
            if '1' in nz[loss_idx+1:] or '1' in cn[loss_idx+1:]:
                sbti_lost_gained_y35 += 1

print(f"Year 1→3: {sbti_lost_y13} lost SBTi, {sbti_lost_gained_y13} gained NZ/CN ({sbti_lost_gained_y13/sbti_lost_y13:.1%})")
print(f"Year 3→5: {sbti_lost_y35} lost SBTi, {sbti_lost_gained_y35} gained NZ/CN ({sbti_lost_gained_y35/sbti_lost_y35:.1%})")

Year 1→3: 5 lost SBTi, 5 gained NZ/CN (100.0%)
Year 3→5: 11 lost SBTi, 10 gained NZ/CN (90.9%)


In [49]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('matrix_fixed.csv')

def analyze_period(df, start, end):
    lost = 0
    gained_nz = 0
    gained_cn = 0
    gained_both = 0
    no_gain = 0
    
    for _, row in df.iterrows():
        sbti = row['sbti'].replace('-', '')[start:end]
        nz = row['nz'].replace('-', '')[start:end]
        cn = row['cn'].replace('-', '')[start:end]
        
        if '1' in sbti:
            idx = sbti.index('1')
            if '0' in sbti[idx+1:]:
                lost += 1
                has_nz = '1' in nz[idx+1:]
                has_cn = '1' in cn[idx+1:]
                
                if has_nz and has_cn:
                    gained_both += 1
                elif has_nz:
                    gained_nz += 1
                elif has_cn:
                    gained_cn += 1
                else:
                    no_gain += 1
    
    return lost, gained_nz, gained_cn, gained_both, no_gain

lost_y13, nz_y13, cn_y13, both_y13, none_y13 = analyze_period(df, 0, 3)
lost_y35, nz_y35, cn_y35, both_y35, none_y35 = analyze_period(df, 2, 5)

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=20,
        thickness=30,
        line=dict(color="white", width=2),
        label=[
            "SBTi Lost<br>2021-2023",
            "SBTi Lost<br>2023-2025",
            "Gained NZ",
            "Gained CN", 
            "Gained Both",
            "No New Target"
        ],
        color=["#D3D3D3", "#D3D3D3", "#00B4B2", "#8B479B", "#4D58A6", "#E5E5E5"],
        x=[0.1, 0.1, 0.9, 0.9, 0.9, 0.9],
        y=[0.2, 0.8, 0.1, 0.4, 0.6, 0.9]
    ),
    link=dict(
        source=[0, 0, 0, 0, 1, 1, 1, 1],
        target=[2, 3, 4, 5, 2, 3, 4, 5],
        value=[nz_y13, cn_y13, both_y13, none_y13, nz_y35, cn_y35, both_y35, none_y35],
        color=["rgba(0,180,178,0.3)", "rgba(139,71,155,0.3)", "rgba(77,88,166,0.3)", "rgba(229,229,229,0.3)"] * 2
    )
)])

fig.update_layout(
    title="SBTi Loss → New Target Adoption (2021-2025)",
    font=dict(size=14),
    height=600,
    width=1200,
    paper_bgcolor='white'
)

fig.write_image("sbti_transitions_periods.png", width=1400, height=700, scale=2)

In [23]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = ["SBTi Committed", "Lost SBTi", "Gained NZ", "Gained CN", "No New Target"],
      color = ["#008AAC", "#D3D3D3", "#00B4B2", "#8B479B", "#E5E5E5"]
    ),
    link = dict(
      source = [0, 1, 1, 1],
      target = [1, 2, 3, 4],
      value = [15, 7, 4, 4],
      color = ["rgba(0,138,172,0.3)", "rgba(0,180,178,0.5)", "rgba(139,71,155,0.5)", "rgba(229,229,229,0.3)"]
  ))])

fig.update_layout(
    title="SBTi Transition Journey: From Validation Loss to New Commitments",
    font_size=12,
    height=400,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)'
)

In [51]:
import pandas as pd
import plotly.graph_objects as go

df = pd.read_csv('matrix_fixed.csv')

def analyze_period(df, start, end):
    lost, gained_nz, gained_cn, gained_both, no_gain = 0, 0, 0, 0, 0
    
    for _, row in df.iterrows():
        sbti = row['sbti'].replace('-', '')[start:end]
        nz = row['nz'].replace('-', '')[start:end]
        cn = row['cn'].replace('-', '')[start:end]
        
        if '1' in sbti:
            idx = sbti.index('1')
            if '0' in sbti[idx+1:]:
                lost += 1
                has_nz = '1' in nz[idx+1:]
                has_cn = '1' in cn[idx+1:]
                
                if has_nz and has_cn:
                    gained_both += 1
                elif has_nz:
                    gained_nz += 1
                elif has_cn:
                    gained_cn += 1
                else:
                    no_gain += 1
    
    return lost, gained_nz, gained_cn, gained_both, no_gain

lost_y13, nz_y13, cn_y13, both_y13, none_y13 = analyze_period(df, 0, 3)
lost_y35, nz_y35, cn_y35, both_y35, none_y35 = analyze_period(df, 2, 5)

fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=25,
        thickness=40,
        line=dict(color="white", width=2),
        label=[
            f"Lost SBTi<br>2021-2023<br>({lost_y13})",
            f"Gained NZ<br>({nz_y13 + nz_y35})",
            f"Gained CN<br>({cn_y13 + cn_y35})",
            f"Gained Both<br>({both_y13 + both_y35})",
            f"No Target<br>({none_y13 + none_y35})",
            f"Lost SBTi<br>2023-2025<br>({lost_y35})"
        ],
        color=["#008AAC", "#00B4B2", "#8B479B", "#4D58A6", "#E5E5E5", "#008AAC"],
        x=[0.1, 0.9, 0.9, 0.9, 0.9, 0.1],
        y=[0.3, 0.1, 0.3, 0.5, 0.7, 0.7]
    ),
    link=dict(
        source=[0, 0, 0, 0, 5, 5, 5, 5],
        target=[1, 2, 3, 4, 1, 2, 3, 4],
        value=[nz_y13, cn_y13, both_y13, none_y13, nz_y35, cn_y35, both_y35, none_y35],
        color=["rgba(0,180,178,0.4)", "rgba(139,71,155,0.4)", "rgba(77,88,166,0.4)", "rgba(229,229,229,0.3)"] * 2
    )
)])

fig.update_layout(
    title="SBTi Transitions: Where Companies Go After Losing Validation",
    font=dict(size=14),
    height=600,
    width=1400,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.write_image("sbti_transitions.png", width=1600, height=700, scale=2)

CN NZ

In [33]:

import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

lost_cn_gained_nz = []
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            if '1' in nz[i+1:]:
                lost_cn_gained_nz.append({'company': row['company'], 'cn': row['cn'], 'nz': row['nz']})
            break

result = pd.DataFrame(lost_cn_gained_nz)
result.to_csv('lost_cn_gained_nz.csv', index=False)
print(result)

lost_cn = 0
for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            break

print(f"Lost CN: {lost_cn}")

                     company     cn     nz
0              América Móvil  10000  01101
1                        NTT  01110  00001
2                      Orlen  -0100  -1011
3             POSCO Holdings  11010  00101
4         Panasonic Holdings  01000  10111
..                       ...    ...    ...
80                     volvo  00010  11101
81  walgreens boots alliance  01000  00010
82                   wistron  001-0  000-1
83              world kinect  --100  --001
84        zf friedrichshafen  11011  00100

[85 rows x 3 columns]
Lost CN: 105


In [34]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

lost_cn = 0
lost_cn_gained_nz = 0

for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            if '1' in nz[i+1:]:
                lost_cn_gained_nz += 1
            break

print(f"Lost CN: {lost_cn}, gained NZ: {lost_cn_gained_nz} ({lost_cn_gained_nz/lost_cn:.1%})")


Lost CN: 105, gained NZ: 85 (81.0%)


In [49]:
import pandas as pd

# Read the 85 successful transitions
successful = pd.read_csv('lost_cn_gained_nz.csv')
successful_companies = set(successful['company'].str.lower().str.strip())

# Read the full matrix to find all companies that lost CN
all_df = pd.read_csv('matrix_fixed.csv')

companies_lost_cn = []
companies_lost_cn_no_nz = []

for _, row in all_df.iterrows():
    company = row['company']
    cn = str(row['cn']).replace('-', '')
    nz = str(row['nz']).replace('-', '')
    
    # Check if lost CN (had 1, then got 0)
    for i in range(len(cn)-1):
        if cn[i] == '1' and '0' in cn[i+1:]:
            companies_lost_cn.append({
                'company': company,
                'cn': row['cn'],
                'nz': row['nz']
            })
            
            # Check if gained NZ
            gained_nz = False
            for j in range(len(nz)-1):
                if nz[j] == '0' and '1' in nz[j+1:]:
                    gained_nz = True
                    break
            
            if not gained_nz:
                companies_lost_cn_no_nz.append({
                    'company': company,
                    'cn': row['cn'],
                    'nz': row['nz']
                })
            break

# Save to CSV
pd.DataFrame(companies_lost_cn).to_csv('all_lost_cn.csv', index=False)
pd.DataFrame(companies_lost_cn_no_nz).to_csv('lost_cn_no_nz.csv', index=False)

print(f"Total lost CN: {len(companies_lost_cn)}")
print(f"Lost CN, no NZ: {len(companies_lost_cn_no_nz)}")
print(f"Lost CN, gained NZ: {len(companies_lost_cn) - len(companies_lost_cn_no_nz)}")


Total lost CN: 105
Lost CN, no NZ: 20
Lost CN, gained NZ: 85


In [46]:
pip install -U kaleido


Note: you may need to restart the kernel to use updated packages.


In [48]:
import plotly.graph_objects as go

fig = go.Figure(go.Waterfall(
    orientation = "v",
    measure = ["relative", "relative", "total"],
    x = ["Companies with CN", "Dropped CN", "Adopted Net Zero"],
    y = [105, -20, 85],
    text = ["105", "-20 (19%)", "85 (81%)"],
    textposition = "outside",
    connector = {"line":{"color":"rgb(63, 63, 63)"}},
    decreasing = {"marker":{"color":"#E5E5E5"}},
    increasing = {"marker":{"color":"#00B4B2"}},
    totals = {"marker":{"color":"#008AAC"}}
))

fig.update_layout(
    title = "Carbon Neutral to Net Zero: 83% Upgrade Rate",
    yaxis_title = "Number of Companies",
    showlegend = False,
    height = 450,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(size=13)
)

# Add annotation box
fig.add_annotation(
    x=2, y=110,
    text="<b>105 companies</b> upgraded<br>to more ambitious targets",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#00B4B2",
    bgcolor="#00B4B2",
    font=dict(color="white", size=12),
    borderpad=10
)


fig.write_image(
    "carbon_neutral_to_net_zero.png",
    format="png",
    scale=2,
    engine="kaleido"
)


C:\Users\AniyaBagheri\AppData\Local\Temp\ipykernel_48536\3233264217.py:39: DeprecationWarning:


Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.




In [37]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

years = ['2021', '2022', '2023', '2024', '2025']

for year_idx, year in enumerate(years):
    cn_count = 0
    nz_count = 0
    both_count = 0
    
    for _, row in df.iterrows():
        cn_char = row['cn'][year_idx]
        nz_char = row['nz'][year_idx]
        
        if cn_char == '1' and nz_char == '1':
            both_count += 1
        elif cn_char == '1':
            cn_count += 1
        elif nz_char == '1':
            nz_count += 1
    
    neither_count = 500 - (cn_count + nz_count + both_count)
    print(f"{year}: CN={cn_count}, NZ={nz_count}, Both={both_count}, Neither={neither_count}")

2021: CN=105, NZ=120, Both=0, Neither=275
2022: CN=103, NZ=182, Both=0, Neither=215
2023: CN=109, NZ=197, Both=0, Neither=194
2024: CN=97, NZ=221, Both=0, Neither=182
2025: CN=89, NZ=251, Both=0, Neither=160


In [48]:
import plotly.graph_objects as go
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

lost_cn = 0
lost_cn_gained_nz = 0

for _, row in df.iterrows():
    cn = row['cn'].replace('-', '')
    nz = row['nz'].replace('-', '')
    
    for i, char in enumerate(cn):
        if char == '1' and '0' in cn[i+1:]:
            lost_cn += 1
            if '1' in nz[i+1:]:
                lost_cn_gained_nz += 1
            break

lost_cn_no_nz = lost_cn - lost_cn_gained_nz

categories = ['CN Dropped', 'NZ Gained', 'CN dropped and not replaced']
values = [-lost_cn, lost_cn_gained_nz, -lost_cn_no_nz]

fig = go.Figure(go.Waterfall(
    x=categories,
    y=values,
    measure=['relative', 'relative', 'total'],
    decreasing={'marker': {'color': 'rgba(139,71,155,0.6)'}},
    increasing={'marker': {'color': 'rgba(0,180,178,0.6)'}},
    totals={'marker': {'color': 'rgba(0,0,0,0.6)'}},
    connector={'line': {'color': 'rgba(100,100,100,0.3)'}},
    text = [
        f"{abs(v)}" if v == lost_cn else f"{abs(v)} ({v / lost_cn:.2f})"
        for v in values
    ],
    textposition='outside',
    width=0.4
))

fig.add_annotation(
    x='NZ Gained',
    y= (lost_cn_gained_nz * -1)+ 80 ,
    text=f"<b>{lost_cn_gained_nz} companies</b> gained NZ<br>after dropping CN",
    showarrow=True,
    arrowhead=2,
    arrowcolor='rgba(0,180,178,0.8)',
    ax=0,
    ay=-80,
    bgcolor='rgba(0,180,178,0.15)',
    bordercolor='rgba(0,180,178,0.8)',
    borderwidth=2,
    borderpad=8,
    font=dict(size=11, color='rgba(0,180,178,0.9)')
)

fig.update_layout(
    yaxis_title='Number of Companies',
    showlegend=False,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis=dict(showgrid=True, gridcolor='rgba(128,128,128,0.2)', zeroline=True)
)
fig.update_traces(width=0.4, selector=dict(type='bar'))
fig.write_image("cn_transitions_waterfall.png", width=600, height=600, scale=2)

print(lost_cn)

105


### sector cluster bubble graphs 

In [67]:
import plotly.graph_objects as go
import numpy as np

sectors = {
    'Technology': ['Accenture', 'Apple', 'Cisco', 'Dell', 'Hon Hai', 'HP', 'LG', 'Microsoft', 'Panasonic', 'SAP', 'Schneider', 'Sony'],
    'Healthcare': ['AstraZeneca', 'Elevance', 'GSK', 'J&J', 'Novartis', 'Pfizer', 'Sanofi'],
    'Retail': ['AEON', 'Ingka', 'Target', 'Tesco', 'Walmart', 'Woolworths'],
    'Finance': ['AmEx', 'ING', 'KB', 'NatWest', 'Visa'],
    'Telecom': ['Deutsche T', 'KDDI', 'SoftBank', 'Telefonica', 'Vodafone'],
    'Food/Bev': ['AB InBev', 'Heineken', 'Nestlé', 'PepsiCo', 'Starbucks'],
    'Auto': ['BMW', 'Continental', 'GM', 'Hyundai'],
    'Transport': ['La Poste'],
    'Industrial': ['ABB'],
    'Household': ['Unilever']
}

colors = ['#00B4B2', '#8B479B', '#4D58A6', '#FF6B6B', '#FFB366', '#66D9B3', '#9966FF', '#FF66B3', '#66B3FF', '#B3FF66']
positions = [(3, 3), (9, 3), (3, 9), (9, 9), (6, 6), (0, 6), (12, 6), (1.5, 0), (6, 0), (10.5, 0)]
font_sizes = [6, 7, 8, 9, 10, 11, 12]

fig = go.Figure()
np.random.seed(999)

for (sector, companies), (cx, cy), color in zip(sectors.items(), positions, colors):
    n = len(companies)
    size = 100 + n * 15
    
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy],
        mode='markers+text',
        marker=dict(size=size, color=color, opacity=0.15, line=dict(width=0)),
        text=f"<b>{sector}</b><br>({n})",
        textposition='middle center',
        textfont=dict(size=16, color=color),
        showlegend=False
    ))
    
    radius = (size / 150) * 0.65
    placed = []
    font_choice = np.random.choice(font_sizes, n)
    
    for company, font_size in zip(companies, font_choice):
        for _ in range(200):
            angle = np.random.uniform(0, 2*np.pi)
            r = np.random.uniform(0.35, 0.85) * radius
            x = cx + r * np.cos(angle)
            y = cy + r * np.sin(angle)
            
            if all(np.sqrt((x-px)**2 + (y-py)**2) > 0.35 for px, py in placed):
                placed.append((x, y))
                fig.add_trace(go.Scatter(
                    x=[x], y=[y],
                    mode='text',
                    text=company,
                    textfont=dict(size=font_size, color=color),
                    showlegend=False
                ))
                break

fig.update_layout(
    title='Sustainability Leaders by Sector',
    xaxis=dict(visible=False, range=[-1, 13]),
    yaxis=dict(visible=False, range=[-1, 11]),
    height=900,
    width=1400,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)'
)

fig.write_image("sector_clusters.png", width=1600, height=1000, scale=2)

### the stacked bar graph for all targets 

In [9]:
import plotly.graph_objects as go
import numpy as np

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
re100 = [50, 60, 73, 75, 74, 77, 78]
sbti = [80, 105, 86, 97, 121, 137, 157]
cn = [50, 85, 105, 102, 111, 98, 91]
nz = [0, 40, 126, 188, 197, 226, 253]
all_actions = [180, 290, 390, 462, 503, 538, 579]

nz_top = np.array(nz)
cn_top = nz_top + np.array(cn)
sbti_top = cn_top + np.array(sbti)
re100_top = sbti_top + np.array(re100)

fig = go.Figure()

fig.add_trace(go.Scatter(x=years, y=all_actions, fill='tozeroy', fillcolor='rgba(255,192,203,0.25)', 
                         line=dict(width=0), showlegend=False, hoverinfo='skip'))

fig.add_trace(go.Bar(x=years, y=nz, name='Net Zero', marker_color='rgba(0,180,178,0.7)', marker_line_width=0))
fig.add_trace(go.Bar(x=years, y=cn, name='Carbon Neutral', marker_color='rgba(139,71,155,0.7)', marker_line_width=0))
fig.add_trace(go.Bar(x=years, y=sbti, name='SBTi', marker_color='rgba(77,88,166,0.7)', marker_line_width=0))
fig.add_trace(go.Bar(x=years, y=re100, name='RE100', marker_color='rgba(159,168,218,0.7)', marker_line_width=0))

fig.update_traces(width=0.4, selector=dict(type='bar'))

bar_width = 0.4
layers = [
    (nz, nz_top, 0, 'rgba(0,180,178,1)', 'nz'),
    (cn, cn_top, nz_top, 'rgba(139,71,155,1)', 'cn'), 
    (sbti, sbti_top, cn_top, 'rgba(77,88,166,1)', 'sbti'),
    (re100, re100_top, sbti_top, 'rgba(159,168,218,1)', 're100')
]

for values, upper_line, lower_line, color, name in layers:
    for i in range(len(years) - 1):
        x_start = years[i] + bar_width/2
        x_end = years[i+1] - bar_width/2
        
        if isinstance(upper_line, np.ndarray):
            upper_y = [upper_line[i], upper_line[i+1]]
        else:
            upper_y = [upper_line, upper_line]
            
        fig.add_trace(go.Scatter(
            x=[x_start, x_end], 
            y=upper_y, 
            mode='lines',
            line=dict(color=color, width=1, dash='dot'),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        if not (name == 'nz' and i == 0):
            pct_change = ((values[i+1] - values[i]) / values[i] * 100)
            
            upper_mid = (upper_y[0] + upper_y[1]) / 2
            
            if isinstance(lower_line, np.ndarray):
                lower_mid = (lower_line[i] + lower_line[i+1]) / 2
            else:
                lower_mid = lower_line
            
            y_position = (upper_mid + lower_mid) / 2
            
            fig.add_annotation(
                x=(years[i] + years[i+1]) / 2,
                y=y_position,
                text=f"{pct_change:+.0f}%",
                showarrow=False,
                font=dict(size=14, color=color),
                textangle=-90
            )

for i, year in enumerate(years):
    fig.add_annotation(
        x=year,
        y=all_actions[i] + 15,
        text=str(all_actions[i]),
        showarrow=False,
        font=dict(size=14, color='#E82C81', family='Open Sans Light')
    )

fig.update_layout(
    barmode='stack',
    xaxis_title='Year',
    yaxis_title='Number of Companies',
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis=dict(showgrid=False)
)

fig.write_image("stacked_growth_with_lines.png", width=1100, height=700, scale=2)

### line graphs 

In [47]:
import plotly.graph_objects as go
import numpy as np

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
re100 = [50, 60, 73, 75, 74, 78, 78]
sbti = [80, 105, 86, 97, 117, 137, 157]
cn = [50, 85, 105, 102, 111, 97, 91]
nz = [0, 40, 126, 188, 200, 225, 253]
all_actions = [180, 290, 390, 462, 503, 538, 579]

fig = go.Figure()

for data, name, color in [
    (nz, 'Net Zero', 'rgba(0,180,178,1.5)'),
    (sbti, 'SBTi', 'rgba(77,88,166,1.5)'),
    (cn, 'Carbon Neutral', 'rgba(139,71,155,1.5)'),
    (re100, 'RE100', 'rgba(159,168,218,1.5)')
]:
    fig.add_trace(go.Scatter(
        x=years, 
        y=data, 
        name=name,
        line=dict(color=color, width=3),
        mode='lines+markers+text',
        marker=dict(size=5),
        text=[f"{v/5:.0f}%" for v in data],
        textposition="top center",
        textfont=dict(size=14, color=color)
    ))

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Companies',
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis=dict(showgrid=False, dtick=50),
    xaxis=dict(showgrid=False)
)

fig.write_image("line_chart_with_percentages.png", width=1100, height=800, scale=2)

In [46]:
import plotly.graph_objects as go

years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
all_actions = [180, 290, 390, 462, 502, 537, 579]
already_due = [25, 55, 39, 32, 27, 35, 43]
by_2030 = [135, 175, 156, 156, 166, 193, 170]
from_2030 = [20, 70, 190, 274, 309, 309, 366]
at_least_one = [120, 165, 249, 302, 321, 346, 362]

fig = go.Figure()

for data, name, color in [
    (all_actions, 'all actions', 'rgba(232, 44, 129, 1.5)'),
    (at_least_one, 'at least one of the actions', 'rgba(228,176,204,1.5)'),
    (from_2030, 'actions due from 2030', 'rgba(255,185,216,1.5)'),
    (by_2030, 'actions due by 2030', 'rgba(220,99,154,1.5)'),
    (already_due, 'actions already due', 'rgba(105,21,59,1.5)')
]:
    fig.add_trace(go.Scatter(
        x=years, 
        y=data, 
        name=name,
        line=dict(color=color, width=3),
        mode='lines+markers+text',
        marker=dict(size=5),
        text=[f"{v}" for v in data],
        textposition="top center",
        textfont=dict(size=14, color=color)
    ))

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Companies',
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    yaxis=dict(showgrid=False,  dtick=50),
    xaxis=dict(showgrid=False)
)

fig.write_image("gf500_climate_action.png", width=1100, height=800, scale=2)

In [53]:
import pandas as pd

all_df = pd.read_csv('matrix_fixed.csv')

# Normalize company names in CSV
all_df['company_clean'] = all_df['company'].str.strip().str.lower()

# Target companies, normalized
target_companies = ["American Express", "ING Group", "KB Financial Group", "NatWest Group", "Visa"]
target_companies_clean = [c.lower() for c in target_companies]

# Filter
filtered_df = all_df[all_df['company_clean'].isin(target_companies_clean)]

print(filtered_df)



                company  re100   sbti     cn     nz      cc  \
51     american express  11111  00001  10000  01111   00111   
307           ing group  11111  01001  10000  01111  0011-1   
330  kb financial group  00111  01111  01000  10111  0-1010   
397       natwest group  ----1  ----1  ----0  ----1   ----1   
577                visa  ---11  ---11  ---10  ---01   ---11   

          company_clean  
51     american express  
307           ing group  
330  kb financial group  
397       natwest group  
577                visa  


In [54]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']

for target in targets:
    suspicious = []
    
    for _, row in df.iterrows():
        company = row['company']
        pattern = row[target]
        
        # Find first 1
        first_one = pattern.find('1')
        if first_one == -1:
            continue
        
        # Look for 0 after first 1 (ignoring -)
        found_zero = False
        zero_pos = first_one + 1
        
        for i in range(first_one + 1, len(pattern)):
            if pattern[i] == '0':
                found_zero = True
                zero_pos = i
                break
        
        if not found_zero:
            continue
        
        # Check if there's a 1 after the 0
        if '1' in pattern[zero_pos + 1:]:
            suspicious.append({
                'company': company,
                'pattern': pattern,
                'issue': f'1 at pos {first_one}, 0 at pos {zero_pos}, 1 again after'
            })
    
    if suspicious:
        pd.DataFrame(suspicious).to_csv(f'flagged_{target}_regained.csv', index=False)
        print(f"{target}: {len(suspicious)} flagged")

re100: 5 flagged
sbti: 3 flagged
cn: 15 flagged
nz: 36 flagged
cc: 78 flagged


In [55]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']

for target in targets:
    suspicious = []
    
    for _, row in df.iterrows():
        company = row['company']
        pattern = row[target]
        
        # Find first 1
        first_one = pattern.find('1')
        if first_one == -1:
            continue
        
        # Look for 0 after first 1 (ignoring -)
        found_zero = False
        zero_pos = -1
        
        for i in range(first_one + 1, len(pattern)):
            if pattern[i] == '0':
                found_zero = True
                zero_pos = i
                break
        
        if not found_zero:
            continue
        
        # Check if there's ANY 1 anywhere after the 0
        for i in range(zero_pos + 1, len(pattern)):
            if pattern[i] == '1':
                suspicious.append({
                    'company': company,
                    'pattern': pattern,
                    'issue': f'1 at {first_one} → 0 at {zero_pos} → 1 at {i}'
                })
                break
    
    if suspicious:
        pd.DataFrame(suspicious).to_csv(f'flagged_{target}_regained.csv', index=False)
        print(f"{target}: {len(suspicious)} flagged")

re100: 5 flagged
sbti: 3 flagged
cn: 15 flagged
nz: 36 flagged
cc: 78 flagged


In [56]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')
targets = ['re100', 'sbti', 'cn', 'nz', 'cc']

for target in targets:
    suspicious = []
    
    for _, row in df.iterrows():
        company = row['company']
        pattern = row[target]
        
        # Find first 1
        first_one = pattern.find('1')
        if first_one == -1:
            continue
        
        # Look for 0 after first 1
        found_zero = False
        zero_pos = -1
        
        for i in range(first_one + 1, len(pattern)):
            if pattern[i] == '0':
                found_zero = True
                zero_pos = i
                break
        
        if not found_zero:
            continue
        
        # Check if there's any 1 after the 0
        for i in range(zero_pos + 1, len(pattern)):
            if pattern[i] == '1':
                suspicious.append({
                    'company': company,
                    'pattern': pattern,
                    'first_1_pos': first_one,
                    'zero_pos': zero_pos,
                    'regained_1_pos': i
                })
                break
    
    if suspicious:
        pd.DataFrame(suspicious).to_csv(f'flagged_{target}_regained.csv', index=False)
        print(f"{target}: {len(suspicious)} companies flagged")
    else:
        print(f"{target}: No suspicious patterns found")

re100: 5 companies flagged
sbti: 3 companies flagged
cn: 15 companies flagged
nz: 36 companies flagged
cc: 78 companies flagged


how many of them were excluded this year / 500
how many of them weer excluded this year /total of the target 
how many of them flagged/ 30 total company entries in historic 

In [ ]:
import pandas as pd

# Load data
df_transitions = pd.read_csv('matrix_fixed.csv')
df_historic = pd.read_csv('Primary_Historic_CSV.csv', encoding='latin-1')

# Get 2025 companies (assuming year 2025 in historic CSV)
companies_2025 = set(df_historic[df_historic['year'] == 2025]['company '].str.strip().str.lower())

targets = ['re100', 'sbti', 'cn', 'nz', 'cc']

print("ERROR MARGIN STATISTICS")
print("="*70)

for target in targets:
    # Load flagged companies
    try:
        flagged_df = pd.read_csv(f'flagged_{target}_regained.csv')
        flagged_companies = set(flagged_df['company'])
        n_flagged = len(flagged_companies)
    except FileNotFoundError:
        print(f"\n{target.upper()}: No flagged file found")
        continue
    
    # Check which flagged companies don't have target in 2025 (last char != '1')
    excluded_this_year = set()
    for company in flagged_companies:
        pattern = df_transitions[df_transitions['company'] == company][target].iloc[0]
        # Remove '-' and check last character
        cleaned = pattern.replace('-', '')
        if cleaned and cleaned[-1] != '1':
            excluded_this_year.add(company)
    
    n_excluded = len(excluded_this_year)
    
    # Total companies with this target in 2025
    target_2025 = df_historic[
        (df_historic['year'] == 2025) & 
        (df_historic[target] == 1)
    ]
    n_target_2025 = len(target_2025)
    
    # Total unique companies in historic
    n_total_historic = df_transitions.shape[0]
    
    print(f"\n{target.upper()}:")
    print(f"  Flagged companies: {n_flagged}")
    print(f"  Excluded in 2025 (last position != 1): {n_excluded}")
    print(f"  Error margins:")
    print(f"    - {n_excluded}/500 Fortune 500 = {n_excluded/500*100:.2f}%")
    print(f"    - {n_excluded}/{n_target_2025} with {target} in 2025 = {n_excluded/n_target_2025*100:.2f}%")
    print(f"    - {n_flagged}/{n_total_historic} historic entries = {n_flagged/n_total_historic*100:.2f}%")

ERROR MARGIN STATISTICS

RE100:
  Flagged companies: 5
  Excluded in 2025 (last position != 1): 0
  Error margins:
    - 0/500 Fortune 500 = 0.00%
    - 0/78 with re100 in 2025 = 0.00%
    - 5/612 historic entries = 0.82%

SBTI:
  Flagged companies: 3
  Excluded in 2025 (last position != 1): 0
  Error margins:
    - 0/500 Fortune 500 = 0.00%
    - 0/157 with sbti in 2025 = 0.00%
    - 3/612 historic entries = 0.49%

CN:
  Flagged companies: 15
  Excluded in 2025 (last position != 1): 3
  Error margins:
    - 3/500 Fortune 500 = 0.60%
    - 3/91 with cn in 2025 = 3.30%
    - 15/612 historic entries = 2.45%

NZ:
  Flagged companies: 36
  Excluded in 2025 (last position != 1): 2
  Error margins:
    - 2/500 Fortune 500 = 0.40%
    - 2/253 with nz in 2025 = 0.79%
    - 36/612 historic entries = 5.88%

CC:
  Flagged companies: 78
  Excluded in 2025 (last position != 1): 9
  Error margins:
    - 9/500 Fortune 500 = 1.80%
    - 9/222 with cc in 2025 = 4.05%
    - 78/612 historic entries = 12.

In [59]:
import pandas as pd

df_matrix = pd.read_csv('matrix_fixed.csv')

leaders = [
    'accenture', 'apple', 'cisco systems', 'dell technologies', 'hon hai precision industry',
    'hp', 'lg electronics', 'microsoft', 'panasonic holdings', 'sap', 'schneider electric', 'sony',
    'astrazeneca', 'elevance health', 'gsk', 'johnson & johnson', 'novartis', 'pfizer', 'sanofi',
    'aeon', 'ingka group', 'target', 'tesco', 'walmart', 'woolworths group',
    'american express', 'ing group', 'kb financial group', 'natwest group', 'visa',
    'deutsche telekom', 'kddi', 'softbank group', 'telefonica', 'vodafone group',
    'anheuser-busch inbev', 'heineken', 'nestlé', 'pepsico', 'starbucks',
    'bmw group', 'continental', 'general motors', 'hyundai mobis',
    'la poste', 'abb', 'unilever'
]

results = []
for leader in leaders:
    match = df_matrix[df_matrix['company'] == leader]
    
    if not match.empty:
        cc_pattern = match['cc'].iloc[0]
        last_char = cc_pattern.replace('-', '')[-1] if cc_pattern.replace('-', '') else 'N/A'
        results.append({
            'company': leader,
            'cc_pattern': cc_pattern,
            'cc_2025': last_char,
            'uses_cc': 'Yes' if last_char == '1' else ('No' if last_char == '-1' else 'Unstated')
        })
    else:
        results.append({'company': leader, 'cc_pattern': None, 'cc_2025': 'Not found', 'uses_cc': 'Not found'})

output = pd.DataFrame(results)
output.to_csv('leaders_cc_usage_2025.csv', index=False)

print(f"Total: {len(leaders)}")
print(f"Yes: {len(output[output['cc_2025'] == '1'])}")
print(f"No: {len(output[output['cc_2025'] == '-1'])}")
print(f"Unstated: {len(output[output['cc_2025'] == '0'])}")

Total: 47
Yes: 28
No: 0
Unstated: 14


In [60]:
import pandas as pd

df = pd.read_csv('matrix_fixed.csv')

# CC Yes: companies with '1' in last position of any target
cc_yes = []
for _, row in df.iterrows():
    company = row['company']
    # Check each target (re100, sbti, cn, nz) for '1' in last position
    for target in ['re100', 'sbti', 'cn', 'nz']:
        pattern = row[target]
        cleaned = pattern.replace('-', '')
        if cleaned and cleaned[-1] == '1':
            cc_yes.append({
                'company': company,
                'target': target,
                'pattern': pattern,
                'cc_pattern': row['cc']
            })
            break  # One entry per company

# CC No: companies with '-1' in CC column last position
cc_no = []
for _, row in df.iterrows():
    cc_pattern = row['cc']
    cleaned = cc_pattern.replace('-', '')
    if cleaned and cleaned[-1] == '-1':
        cc_no.append({
            'company': row['company'],
            'cc_pattern': cc_pattern,
            're100': row['re100'],
            'sbti': row['sbti'],
            'cn': row['cn'],
            'nz': row['nz']
        })

pd.DataFrame(cc_yes).to_csv('cc_yes_with_targets.csv', index=False)
pd.DataFrame(cc_no).to_csv('cc_no_with_targets.csv', index=False)

print(f"CC Yes: {len(cc_yes)}")
print(f"CC No: {len(cc_no)}")

CC Yes: 429
CC No: 0
